# MODELO SARIMA – PROJEÇÃO DO IPCA

## Modelagem da Inflação com SARIMA

O IPCA (Índice de Preços ao Consumidor Amplo) é o principal indicador oficial de inflação no Brasil, sendo amplamente utilizado como referência para metas de política monetária e análises macroeconômicas.

Por se tratar de uma série temporal mensal, observa-se a presença de:

- Componentes autoregressivos  
- Tendência de médio prazo  
- Sazonalidade anual recorrente (período de 12 meses)  

---

## Estrutura da Inflação Brasileira

A inflação brasileira apresenta padrões sazonais associados a:

- Reajustes de preços administrados  
- Ciclos de consumo das famílias  
- Efeitos climáticos sobre alimentos  
- Períodos específicos de maior demanda (ex.: início de ano)  

Esses fatores reforçam a necessidade de um modelo capaz de capturar simultaneamente dinâmica temporal e sazonalidade.

---

## Justificativa do Modelo SARIMA

Dessa forma, opta-se pela utilização do modelo **SARIMA**, que permite capturar:

1. Dinâmica de curto prazo (componente ARIMA não sazonal)  
2. Estrutura sazonal explícita  

---

## Especificação Geral
O modelo SARIMA amplia o ARIMA tradicional ao incorporar termos sazonais, tornando-se mais adequado para séries com periodicidade definida e comportamento cíclico recorrente.
Especificação Geral:

$SARIMA(p,d,q)(P,D,Q,12)$

Onde:

- **(p,d,q)** → componente não sazonal  
  - p: ordem autoregressiva  
  - d: grau de diferenciação  
  - q: ordem da média móvel  

- **(P,D,Q)** → componente sazonal  
  - P: ordem autoregressiva sazonal  
  - D: diferenciação sazonal  
  - Q: média móvel sazonal  

- **12** → periodicidade anual (dados mensais)

---

## Critérios para Escolha dos Parâmetros

A escolha dos parâmetros será orientada por:

- Teste de estacionariedade (ADF – Augmented Dickey-Fuller)  
- Análise das funções (ACF)
- Análise das funções de autocorrelação parcial (PACF) 
- Critérios de informação (AIC e BIC)  

---

## Objetivo

Obter um modelo estatisticamente consistente e capaz de gerar projeções confiáveis do IPCA, respeitando sua estrutura temporal e sazonal.

## TESTE ADF

In [ ]:
def testar_estacionariedade_ipca(serie, tipo='ct'):
    """
    Teste de Estacionariedade via Dickey-Fuller Aumentado (ADF)

    Aplicado à série do IPCA.

    Objetivo:
    Verificar a presença de raiz unitária na série temporal,
    condição que indica não estacionariedade.

    Hipóteses do Teste:
    H0: A série possui raiz unitária (não estacionária)
    H1: A série é estacionária
    Regra de Decisão:
    Se p-valor < 0.05:
        Rejeita-se H0 → Série estacionária.
    Caso contrário:
        Não se rejeita H0 → Série não estacionária.

    Parâmetros:
    serie : pandas.Series
        Série temporal do IPCA.

    tipo : str
        'c'  → constante Assume que a série oscila em torno de uma média fixa.
        'ct' → constante + tendência Assume que a série possui uma tendência linear.
        'nc' → sem constante Se Usa quando a série oscila em torno de zero.
        
    Observação:
    A estacionariedade é pressuposto fundamental
    para aplicação de modelos ARIMA/SARIMA.
    Caso a série não seja estacionária,
    aplica-se diferenciação (parâmetro 'd')
    """

    serie_limpa = serie.dropna()

    resultado = adfuller(serie_limpa, regression=tipo)

    estatistica = resultado[0]
    p_valor = resultado[1]
    lags = resultado[2]
    n_obs = resultado[3]
    valores_criticos = resultado[4]

    print("="*60)
    print("TESTE ADF - IPCA")
    print("="*60)

    print(f"Tipo de regressão: {tipo}")
    print(f"Estatística ADF: {estatistica}")
    print(f"p-valor: {p_valor}")
    print(f"Lags utilizados: {lags}")
    print(f"Número de observações: {n_obs}")

    print("\nValores críticos:")
    for chave, valor in valores_criticos.items():
        print(f"{chave}: {valor}")

    # GRÁFICO

    labels = [
        "Crítico 1%",
        "Crítico 5%",
        "Crítico 10%",
        "ADF"
    ]

    valores = [
        valores_criticos["1%"],
        valores_criticos["5%"],
        valores_criticos["10%"],
        estatistica
    ]

    plt.figure(figsize=(7,4))

    barras = plt.bar(labels, valores)

    plt.title("Teste ADF: IPCA")
    plt.ylabel("Valor da estatística")

    for barra in barras:
        altura = barra.get_height()
        plt.text(
            barra.get_x() + barra.get_width()/2,
            altura,
            f"{altura:.2f}",
            ha='center',
            va='bottom'
        )

    plt.show()

    # DECISÃO

    if p_valor < 0.05:
        print("Conclusão: Rejeita-se H0 → Série estacionária")
        return True
    else:
        print("Conclusão: Não se rejeita H0 → Série não estacionária")
        return False

## Definição dos Parâmetros do SARIMA
$$
SARIMA(p,d,q)(P,D,Q,12)
$$

# Definição de d (diferenciação não sazonal)
- Tornar a série estacionária em nível.
- Aplicando o Teste ADF:
- Se p-valor < 0.05 → d = 0
- Se p-valor > 0.05 → d = 1

# Definição de D (diferenciação sazonal)
## Mesmo que a série seja estacionária em nível, ela pode ter raiz unitária sazonal.
### Como identificar?

#### Quando:
- A série já é estável ao longo dos ciclos anuais
- A ACF não mostra picos fortes em lag 12
- A diferença sazonal não melhora o comportamento
- ENTÃO D = 0

#### Quando:
- Sazonalidade anual forte e persistente
- ACF elevada em múltiplos de 12
- Evidência de raiz unitária sazonal
- Então aplicamos: $Y_t - Y_{t-12}$
- ENTÃO D = 1

# Definição de p e q (parte não sazonal)
##  ACF e PACF
- AR (p) → a série depende dela mesma (O valor de hoje depende diretamente do valor de ontem.)
- MA (q) → a série depende dos erros passados (O valor de hoje depende do erro que aconteceu ontem.)
- ACF (Autocorrelation Function) $YteYt−k$ Ela mede dependência total (direta + indireta)
- PACF Mede a correlação direta, removendo efeitos intermediários. Ela responde se Existe relação direta entre $YteYt−k$
### Como identificar p (AR)?
- Se a PACF: Tem pico forte no lag 1 e depois zera então → p = 1
- Se a PACF: Tem picos fortes até lag 2 e depois zera então → p = 2
### Como identificar q (MA)?
- Se a ACF: Tem pico forte no lag 1 e depois corta então existe 1 erro passado influenciando → q = 1
- Se a ACF: Tem pico forte no lag 2 e depois corta então fica dentro da banda de confiança, isso sugere → q = 2

# Definição de P e Q (parte sazonal)
## ACF e PACF nas defasagens múltiplas de 12
- Se PACF tem pico forte em lag 12 → P = 1
- Se ACF tem pico forte em lag 12 → Q = 1
- Não houve AR sazonal relevante → P = 0
- Houve MA sazonal → Q = 1

# Definição de s
| Frequência     | s   |
| -------------- | --- |
| Mensal         | 12  |
| Trimestral     | 4   |
| Semanal        | 52  |
| Diário (anual) | 365 |



In [ ]:
def ajustar_sarima_auto(serie, d=0, D=0, s=12):
    """
    Ajuste automático do modelo SARIMA com seleção baseada no Critério de Informação de Akaike (AIC).

    Fundamentação Metodológica:
    O modelo SARIMA (Seasonal AutoRegressive Integrated Moving Average)
    constitui uma extensão do modelo ARIMA tradicional, incorporando
    componentes sazonais explícitos capazes de capturar padrões periódicos
    recorrentes na série temporal.

    Sua especificação geral é dada por:

        SARIMA(p,d,q)(P,D,Q,s)

    Onde:

    Parte Não Sazonal:
        p → número de defasagens autoregressivas (AR),
            capturando dependência da série em relação aos próprios valores passados.

        d → grau de diferenciação não sazonal,
            aplicado quando a série apresenta raiz unitária
            (não estacionariedade em nível).

        q → número de termos de média móvel (MA),
            modelando a influência de choques passados.

    Parte Sazonal:
        P → componente autoregressivo sazonal;
        D → grau de diferenciação sazonal;
        Q → componente de média móvel sazonal;
        s → periodicidade da sazonalidade (12 para dados mensais)

    Estratégia de Seleção:
    A escolha da melhor especificação é realizada por meio da comparação
    do Critério de Informação de Akaike (AIC), definido como:

        AIC = -2 log(L) + 2k

    onde:
        L = função de verossimilhança do modelo estimado
        k = número de parâmetros estimados

    O AIC penaliza modelos excessivamente complexos,
    buscando equilíbrio entre qualidade do ajuste e parcimônia.

    Procedimento Computacional:
    1. Define-se um espaço restrito de busca para os parâmetros (p,q,P,Q),
       evitando superparametrização e instabilidade numérica.
    2. Cada combinação é estimada via Máxima Verossimilhança,
       utilizando o algoritmo de otimização 'Powell',
       mais robusto em contextos de séries macroeconômicas.
    3. Apenas modelos que convergirem adequadamente são considerados.
    4. O modelo com menor AIC é selecionado como especificação final.

    Observações Técnicas:
    enforce_stationarity=False:
        Relaxa restrições formais durante a estimação,
        permitindo maior flexibilidade na busca do ótimo.

    enforce_invertibility=False:
        Evita falhas de convergência associadas à parte MA.

    Retorno:
    melhor_modelo : objeto SARIMAXResults
        Modelo ajustado com menor AIC.
    df_resultados : DataFrame
        Tabela contendo todas as especificações testadas
        e seus respectivos critérios de informação.
    """
   
    # Espaço de busca ampliado para maior robustez
    p = range(0, 3)   # 0,1,2
    q = range(0, 3)   # 0,1,2
    P = range(0, 2)   # 0,1
    Q = range(0, 2)   # 0,1
    
    parametros = list(itertools.product(p, q, P, Q))
    
    melhor_aic = float("inf")
    melhor_modelo = None
    melhor_param = None
    
    resultados = []
    
    for param in parametros:
        p_, q_, P_, Q_ = param
        
        try:
            modelo = SARIMAX(
                serie,
                order=(p_, d, q_),
                seasonal_order=(P_, D, Q_, s),
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            
            resultado = modelo.fit(method='powell', disp=False)

            # Considera apenas modelos que convergiram
            if resultado.mle_retvals['converged']:
                
                aic = resultado.aic
                
                resultados.append({
                    'p': p_, 'd': d, 'q': q_,
                    'P': P_, 'D': D, 'Q': Q_,
                    's': s,
                    'AIC': aic,
                    'BIC': resultado.bic
                })

                 # Atualiza melhor modelo
                if aic < melhor_aic:
                    melhor_aic = aic
                    melhor_modelo = resultado
                    melhor_param = (p_, d, q_, P_, D, Q_, s)
                    
        except:
            continue
    
    df_resultados = pd.DataFrame(resultados).sort_values("AIC")
    
    print("Melhor modelo encontrado:")
    print(f"SARIMA{melhor_param}")
    print(f"AIC: {melhor_aic}")
    print("\nResumo do modelo:")
    print(melhor_modelo.summary())
    
    return melhor_modelo, df_resultados

## Previsão do IPCA

In [ ]:
def prever_ipca(modelo, passos=41):
    """
    Função para geração de previsões fora da amostra (out-of-sample)
    para o IPCA a partir de um modelo SARIMAX previamente estimado.

    Fundamentação metodológica:
    Após a estimação do modelo SARIMA/SARIMAX por máxima verossimilhança,
    é possível obter projeções condicionais à informação disponível até
    o último período observado. Essas previsões são baseadas na estrutura
    dinâmica estimada (componentes AR, MA e sazonais).

    Parâmetros
    modelo : objeto SARIMAXResults
        Modelo previamente ajustado aos dados históricos do IPCA.
        Pressupõe-se que o modelo já foi validado via diagnóstico
        de resíduos (ruído branco, ausência de autocorrelação e
        normalidade aproximada).

    passos : int, default=48
        Horizonte de previsão em meses.
        Exemplo: 48 passos correspondem a uma projeção de 4 anos
        à frente, considerando periodicidade mensal.

    Retorno
    df_previsao : DataFrame
        DataFrame contendo:
        - ipca_previsto       → previsão pontual (média condicional)
        - limite_inferior     → limite inferior do intervalo de confiança
        - limite_superior     → limite superior do intervalo de confiança

    Interpretação Econômica
    A previsão pontual representa o valor esperado do IPCA condicional
    à informação disponível no período t.

    Os intervalos de confiança refletem a incerteza associada ao processo
    estocástico estimado, incorporando a variância dos erros e a propagação
    dessa variância ao longo do horizonte de previsão.

    Observação:
    Quanto maior o horizonte (passos), maior tende a ser a amplitude
    dos intervalos de confiança, devido ao acúmulo de incerteza.
    """

    # Geração da previsão condicional ao conjunto de informações disponível
    previsao = modelo.get_forecast(steps=passos)

    # Extração da média prevista (previsão pontual)
    media = previsao.predicted_mean

    # Extração dos intervalos de confiança (default: 95%)
    intervalo = previsao.conf_int()

    # Organização dos resultados em DataFrame estruturado
    df_previsao = media.to_frame(name='ipca_previsto')
    df_previsao['limite_inferior'] = intervalo.iloc[:, 0]
    df_previsao['limite_superior'] = intervalo.iloc[:, 1]

    return df_previsao

## Plotar a Previsão

In [ ]:
def plotar_previsao(serie, df_previsao):

    """
    Função para visualização da série histórica do IPCA e
    das projeções geradas pelo modelo SARIMA, incluindo
    os respectivos intervalos de previsão.
    """

    # ============================================================
    # GRÁFICO
    # ============================================================

    plt.figure(figsize=(12, 6))

    # Série histórica observada
    plt.plot(
        serie.index,
        serie.values,
        label='IPCA Observado'
    )

    # Previsão pontual
    plt.plot(
        df_previsao.index,
        df_previsao['ipca_previsto'],
        label='IPCA Previsto'
    )

    # Intervalo de previsão
    plt.fill_between(
        df_previsao.index,
        df_previsao['limite_inferior'],
        df_previsao['limite_superior'],
        alpha=0.3,
        label='Intervalo de Previsão (95%)'
    )

    plt.legend()
    plt.title('Projeção do IPCA via Modelo SARIMA')
    plt.xlabel('Período')
    plt.ylabel('IPCA (%)')
    plt.grid(True)

    plt.tight_layout()
    plt.show()

    # ============================================================
    # RESUMO NUMÉRICO DA PROJEÇÃO
    # ============================================================

    primeiro = df_previsao.iloc[0]
    ultimo = df_previsao.iloc[-1]

    data_min = df_previsao['ipca_previsto'].idxmin()
    data_max = df_previsao['ipca_previsto'].idxmax()

    print("\n" + "=" * 60)
    print("RESUMO DA PROJEÇÃO DO IPCA")
    print("=" * 60)

    print(
        f"\nPrimeiro período projetado: "
        f"{df_previsao.index[0].strftime('%m/%Y')}"
    )
    print(
        f"IPCA previsto: "
        f"{primeiro['ipca_previsto']:.4f}%"
    )
    print(
        f"Limite inferior (95%): "
        f"{primeiro['limite_inferior']:.4f}%"
    )
    print(
        f"Limite superior (95%): "
        f"{primeiro['limite_superior']:.4f}%"
    )

    print(
        f"\nÚltimo período projetado: "
        f"{df_previsao.index[-1].strftime('%m/%Y')}"
    )
    print(
        f"IPCA previsto: "
        f"{ultimo['ipca_previsto']:.4f}%"
    )
    print(
        f"Limite inferior (95%): "
        f"{ultimo['limite_inferior']:.4f}%"
    )
    print(
        f"Limite superior (95%): "
        f"{ultimo['limite_superior']:.4f}%"
    )

    print(
        f"\nMenor previsão pontual: "
        f"{df_previsao.loc[data_min, 'ipca_previsto']:.4f}% "
        f"em {data_min.strftime('%m/%Y')}"
    )

    print(
        f"Maior previsão pontual: "
        f"{df_previsao.loc[data_max, 'ipca_previsto']:.4f}% "
        f"em {data_max.strftime('%m/%Y')}"
    )